[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niccoloridi/International-Law-and-AI/blob/main/IntlLaw_AI_Lab04_Feeding_the_Machine.ipynb)

In [ ]:
#@title Lab 04 — Feeding the Machine
print("""
 █████             █████    █████                                                     █████████   █████
░░███             ░░███    ░░███                                         ███         ███░░░░░███ ░░███ 
 ░███  ████████   ███████   ░███         ██████   █████ ███ █████       ░███        ░███    ░███  ░███ 
 ░███ ░░███░░███ ░░░███░    ░███        ░░░░░███ ░░███ ░███░░███     ███████████    ░███████████  ░███ 
 ░███  ░███ ░███   ░███     ░███         ███████  ░███ ░███ ░███    ░░░░░███░░░     ░███░░░░░███  ░███ 
 ░███  ░███ ░███   ░███ ███ ░███      █ ███░░███  ░░███████████         ░███        ░███    ░███  ░███ 
 █████ ████ █████  ░░█████  ███████████░░████████  ░░████░████          ░░░         █████   █████ █████
░░░░░ ░░░░ ░░░░░    ░░░░░  ░░░░░░░░░░░  ░░░░░░░░    ░░░░ ░░░░                      ░░░░░   ░░░░░ ░░░░░ 
                                                                                                       
                                                                                                       
                                                                                                       

   Lab 04 // Feeding the Machine
   Melbourne Law Masters 2026
""")

*Brought to you by **Dr Niccolò Ridi**, Purveyor of Fine Scripts* - [KCL Profile](https://www.kcl.ac.uk/people/niccolo-ridi)

# Feeding the Machine: Annotation, Labour, and the Politics of AI Alignment

In this lab, you will become a data annotator. You'll label text passages for sentiment and toxicity, then re-label the same texts under different instructions. Your annotations will feed directly into a machine learning pipeline, demonstrating in real time how human labelling decisions shape AI behaviour.

**This is RLHF in action**: Reinforcement Learning from Human Feedback. But beneath the technical elegance lies a critical question for international law: *Who decides what counts as toxic, relevant, or dangerous? And what are the labour and governance implications?*

## Table of Contents

> **Part One: The Human Behind the Data**  
> > [The Invisible Labour of Annotation](#scrollTo=part-one-heading)  
> > [Your Dataset](#scrollTo=dataset-intro)

> **Part Two: Annotate! (Round 1)**  
> > [Annotation Interface](#scrollTo=round-one-heading)  
> > [Label Distribution](#scrollTo=round-one-viz)

> **Part Three: Re-Annotate! (Round 2 – Changed Instructions)**  
> > [The Same Texts, Different Frames](#scrollTo=round-two-heading)  
> > [Inter-Annotator Disagreement](#scrollTo=disagreement-viz)

> **Part Four: From Labels to Model**  
> > [The Fine-Tuning Loop](#scrollTo=finetuning-heading)  
> > [Model Predictions Before and After](#scrollTo=model-predictions)

> **Part Five: The Politics of Labelling**  
> > [Critical Reflection](#scrollTo=reflection-heading)  
> > [Whose Values?](#scrollTo=whose-values)

> **Review Quiz**  
> > [Test Your Understanding](#scrollTo=quiz-heading)

---

# Part One: The Human Behind the Data {#part-one-heading}

### The Invisible Labour of Annotation

In 2023, TIME magazine revealed that OpenAI hired contractors in Kenya to label toxic content for ChatGPT – often earning less than $2 USD per hour. They read graphic descriptions of violence, sexual abuse, and extremist propaganda to train AI systems that would then be used by billions.

This is data annotation: the hidden labour behind every "aligned" AI system.

**Key readings you should return to:**
- Kate Crawford, *Atlas of AI* (2021) – on the material and labour dimensions of machine learning
- Aaron Cant, *Feeding the Machine* – on digital labour platforms and data annotation
- Time: "OpenAI Used Kenyan Workers on Poverty Wages to Make ChatGPT Less Toxic"

**Why this matters for international law:**
- **Labour rights** (ICESCR Articles 6-7): Who are annotators? What are their working conditions? What is their "right to work"?
- **Governance**: Annotation guidelines embed values. Who decides what counts as "toxic"? This IS a form of governance.
- **Digital colonialism**: AI development is concentrated in wealthy countries; annotation labour is outsourced to lower-income countries.
- **Algorithmic justice**: The AI systems trained on this labour are deployed in legal, medical, and security contexts – potentially reinforcing existing inequalities.

---

**In this lab, you will annotate. You will also disagree with yourself. You will see how tiny changes in instruction lead to different labels. Imagine this at scale: millions of annotators, billions of labels, each one shaping the AI systems that will govern our digital futures.**

In [ ]:
#@title ## Load libraries and prepare the dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

### Your Dataset {#dataset-intro}

Below are 20 text passages covering international affairs, conflict, politics, and legal matters. They are deliberately chosen to be **ambiguous, multi-faceted, and context-dependent**. This is realistic: most real-world data is messy and resists clean categorization.

Read each one carefully. Notice how your *interpretation* depends on your background, values, and the framing you've been given.

In [ ]:
#@title ## Dataset: 20 Texts for Annotation

texts = [
    "The international community must stand united against human rights violations.",
    "Trade sanctions are a form of economic warfare that harm civilian populations.",
    "Freedom of speech is an absolute right, even for those we disagree with.",
    "Refugees fleeing persecution deserve protection under international law.",
    "State sovereignty is paramount and supranational bodies threaten national independence.",
    "The rules-based international order has failed developing nations for decades.",
    "Military intervention is sometimes justified to prevent mass atrocities.",
    "Colonial powers must pay reparations for centuries of exploitation.",
    "Digital surveillance by governments is essential for national security.",
    "Big Tech companies should be regulated like public utilities.",
    "Indigenous peoples have inherent rights to their ancestral lands.",
    "Climate change is the greatest threat to human rights and peace.",
    "Corporations are using cheap labour in poorer countries to maximize profits.",
    "International law is a tool of powerful nations to maintain hegemony.",
    "The death penalty violates fundamental human dignity and must be abolished globally.",
    "Whistleblowers who leak classified documents are traitors to their nations.",
    "Children in conflict zones face unimaginable suffering and trauma.",
    "Tax havens enable the wealthy to evade obligations to society.",
    "The International Court of Justice lacks enforcement power and is symbolic only.",
    "Data is the new oil, and its extraction mirrors colonial resource exploitation."
]

# Create initial dataframe
df_dataset = pd.DataFrame({
    'id': range(1, 21),
    'text': texts
})

print(f"Dataset: {len(texts)} texts loaded\n")
print(df_dataset.to_string(index=False))

---

# Part Two: Annotate! (Round 1) {#round-one-heading}

### Your Task: Annotation Round 1

You are now a data annotator. You have been given the following task:

**Your job: For each text passage, provide three pieces of feedback:**

1. **Sentiment**: Is the overall tone of the text Positive, Neutral, or Negative?
   - *Positive*: Hopeful, constructive, solution-oriented
   - *Neutral*: Factual, descriptive, balanced
   - *Negative*: Critical, adversarial, blame-oriented

2. **Toxicity Score** (1-5): How harmful, divisive, or inflammatory is this text?
   - *1*: Completely neutral, no inflammatory language
   - *3*: Moderate – some strong language or divisive framing
   - *5*: Highly toxic – dehumanizing, extremist, or violently adversarial

3. **Relevance to International Law**: Is this statement directly relevant to international law?
   - *Yes*: Directly invokes treaties, international institutions, or cross-border governance
   - *No*: Purely domestic or non-legal
   - *Unclear*: Could go either way

---

**Click below to enter your annotations.**

## Annotation Interface: Round 1

📋 ANNOTATION ROUND 1 INSTRUCTIONS

Below, you will enter your annotations as comma-separated values.
Format for each line: SENTIMENT, TOXICITY, RELEVANCE

Example: Negative, 4, Yes

You can run this cell with your annotations, or use the
form-based approach below.

In [ ]:
#@title ## Round 1: Sentiment Annotations (Enter your labels)
#@markdown For each text (1-20), enter sentiment: Positive / Neutral / Negative

# Simulated annotations for demonstration (students would enter their own)
round1_sentiments = "Positive, Neutral, Positive, Positive, Negative, Negative, Negative, Positive, Negative, Negative, Positive, Positive, Negative, Negative, Positive, Negative, Negative, Positive, Negative, Negative".split(", ")

round1_toxicity = [1, 3, 2, 1, 4, 4, 4, 3, 4, 3, 2, 2, 3, 5, 1, 4, 1, 2, 2, 3]

round1_relevance = "Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes".split(", ")

# Create annotation dataframe
df_round1 = df_dataset.copy()
df_round1['sentiment_r1'] = round1_sentiments
df_round1['toxicity_r1'] = round1_toxicity
df_round1['relevance_r1'] = round1_relevance

print("\n✓ Round 1 annotations recorded!\n")
print(df_round1[['id', 'sentiment_r1', 'toxicity_r1', 'relevance_r1']].to_string(index=False))

### Label Distribution: What Did You Label? {#round-one-viz}

In [ ]:
#@title ## Visualize Round 1 Annotations

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sentiment distribution
sentiment_counts = df_round1['sentiment_r1'].value_counts()
axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=['#2ecc71', '#95a5a6', '#e74c3c'])
axes[0].set_title('Sentiment Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

# Toxicity distribution
axes[1].hist(df_round1['toxicity_r1'], bins=5, color='#e67e22', edgecolor='black', alpha=0.7)
axes[1].set_title('Toxicity Score Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Toxicity Score (1-5)')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y', alpha=0.3)

# Relevance distribution
relevance_counts = df_round1['relevance_r1'].value_counts()
axes[2].bar(relevance_counts.index, relevance_counts.values, color=['#3498db', '#95a5a6', '#9b59b6'])
axes[2].set_title('Relevance to Intl Law', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Count')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('round1_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n📊 Summary Statistics:\n")
print(f"Mean Toxicity Score: {df_round1['toxicity_r1'].mean():.2f}")
print(f"Median Toxicity Score: {df_round1['toxicity_r1'].median():.2f}")
print(f"Most common sentiment: {sentiment_counts.index[0]} ({sentiment_counts.values[0]} texts)")
print(f"\nPositive: {(df_round1['sentiment_r1'] == 'Positive').sum()} texts")
print(f"Neutral: {(df_round1['sentiment_r1'] == 'Neutral').sum()} texts")
print(f"Negative: {(df_round1['sentiment_r1'] == 'Negative').sum()} texts")

---

# Part Three: Re-Annotate! (Round 2 – Changed Instructions) {#round-two-heading}

### Now We Change the Rules

You just annotated 20 texts. Now, here's the twist:

**You will annotate the SAME texts again. But this time, with DIFFERENT instructions.**

This time, imagine you are annotating for two different audiences:

---

### Scenario A: Government Perspective

*You are a data analyst for a government security agency. Your job is to identify content that undermines state sovereignty, challenges government authority, or foments dissent.*

**For this perspective:**
- **Toxicity now means**: Anti-state rhetoric, undermining national interests, calls for sovereignty restrictions
- Statements criticizing national governments, promoting supranational governance, or questioning state authority should score HIGH on toxicity

---

### Scenario B: Human Rights Advocate Perspective

*You are an analyst for an international human rights NGO. Your job is to identify content that violates human dignity, promotes discrimination, or normalizes violence.*

**For this perspective:**
- **Toxicity now means**: Dehumanizing language, discrimination, violence-justification, structural oppression
- Statements that normalize inequality, defend oppressive systems, or minimize human suffering should score HIGH on toxicity

---

**Here's the critical insight: SAME TEXTS, DIFFERENT FRAMES. Your labels will diverge from Round 1 – perhaps dramatically.**

This is what **inter-annotator disagreement** looks like. This is why "ground truth" is a myth.

In [ ]:
#@title ## Round 2: Re-Annotation with Changed Instructions
#@markdown Select which perspective to adopt:

perspective = "Human Rights Advocate" #@param ["Government Official", "Human Rights Advocate"]

print(f"\n🎭 You are now: {perspective}")
print("="*60)

if perspective == "Government Official":
    print("\nYour task: Identify anti-state, sovereignty-challenging content.")
    print("High toxicity = threats to national interests & authority.\n")
    # Simulated government official annotations
    round2_toxicity = [2, 5, 3, 4, 1, 5, 4, 5, 2, 4, 4, 5, 4, 5, 3, 2, 3, 4, 1, 5]
else:
    print("\nYour task: Identify dehumanizing, oppressive, harmful content.")
    print("High toxicity = violations of human dignity & rights.\n")
    # Simulated human rights advocate annotations
    round2_toxicity = [1, 2, 1, 1, 5, 2, 4, 2, 5, 2, 1, 1, 4, 5, 1, 5, 1, 2, 3, 3]

df_round2 = df_round1.copy()
df_round2['toxicity_r2'] = round2_toxicity

print("✓ Round 2 annotations recorded!\n")
print("Comparing Round 1 vs Round 2 toxicity scores:")
print()
comparison = pd.DataFrame({
    'Text ID': df_round2['id'],
    'Round 1 Toxicity': df_round2['toxicity_r1'],
    'Round 2 Toxicity': df_round2['toxicity_r2'],
    'Δ (Difference)': df_round2['toxicity_r2'] - df_round2['toxicity_r1']
})
print(comparison.to_string(index=False))

### Inter-Annotator Disagreement {#disagreement-viz}

In [ ]:
#@title ## Visualize the Disagreement

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Scatter plot of Round 1 vs Round 2
axes[0, 0].scatter(df_round2['toxicity_r1'], df_round2['toxicity_r2'], s=100, alpha=0.6, color='#3498db')
axes[0, 0].plot([0, 5], [0, 5], 'r--', label='Perfect agreement', linewidth=2)
axes[0, 0].set_xlabel('Round 1 Toxicity', fontsize=11)
axes[0, 0].set_ylabel('Round 2 Toxicity', fontsize=11)
axes[0, 0].set_title('Round 1 vs Round 2 Toxicity Scores', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[0, 0].set_xlim(0, 5.5)
axes[0, 0].set_ylim(0, 5.5)

# Plot 2: Difference distribution
differences = df_round2['toxicity_r2'] - df_round2['toxicity_r1']
axes[0, 1].bar(range(len(differences)), differences, color=['#e74c3c' if d > 0 else '#2ecc71' if d < 0 else '#95a5a6' for d in differences])
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[0, 1].set_xlabel('Text ID', fontsize=11)
axes[0, 1].set_ylabel('Toxicity Change (Δ)', fontsize=11)
axes[0, 1].set_title('Magnitude of Disagreement per Text', fontsize=12, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Absolute difference distribution
abs_diff = abs(differences)
axes[1, 0].hist(abs_diff, bins=5, color='#9b59b6', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Absolute Difference |Δ|', fontsize=11)
axes[1, 0].set_ylabel('Count', fontsize=11)
axes[1, 0].set_title('Distribution of Disagreement Magnitude', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

# Plot 4: Text with largest disagreement
max_diff_idx = abs_diff.idxmax()
max_diff_text = df_round2.loc[max_diff_idx]
axes[1, 1].axis('off')
text_info = f"""LARGEST DISAGREEMENT:

Text #{int(max_diff_text['id'])}:
\\"{max_diff_text['text']}\\"

Round 1 Score: {max_diff_text['toxicity_r1']}
Round 2 Score: {max_diff_text['toxicity_r2']}
Difference: {max_diff_text['toxicity_r2'] - max_diff_text['toxicity_r1']:+.0f}

This text is POLARIZING.
Your interpretation changed dramatically
based on the framing and instructions.
"""
axes[1, 1].text(0.1, 0.5, text_info, fontsize=10, verticalalignment='center',
                bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8),
                family='monospace')

plt.tight_layout()
plt.savefig('round1_vs_round2.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n📊 Disagreement Statistics:\n")
print(f"Mean absolute disagreement: {abs_diff.mean():.2f}")
print(f"Max disagreement: {abs_diff.max():.0f}")
print(f"Texts with NO disagreement (Δ=0): {(abs_diff == 0).sum()}")
print(f"Texts with HIGH disagreement (|Δ|≥2): {(abs_diff >= 2).sum()}")

In [ ]:
#@title ## Calculate Inter-Annotator Agreement
#@markdown Using simple percent agreement and Cohen's kappa (approx)

from sklearn.metrics import cohen_kappa_score

# Convert to ordinal scale for kappa
# Kappa expects integer categories, so we use toxicity scores directly
try:
    kappa = cohen_kappa_score(df_round2['toxicity_r1'], df_round2['toxicity_r2'])
except:
    kappa = np.nan

# Simple percent agreement (allowing ±1 as "close")
close_agreement = (abs_diff <= 1).sum()
perfect_agreement = (abs_diff == 0).sum()

print("\n📈 Inter-Annotator Agreement Metrics:\n")
print(f"Perfect agreement (Δ=0): {perfect_agreement}/20 ({perfect_agreement/20*100:.1f}%)")
print(f"Close agreement (|Δ|≤1): {close_agreement}/20 ({close_agreement/20*100:.1f}%)")
print(f"Cohen's Kappa: {kappa:.3f}" if not np.isnan(kappa) else "Cohen's Kappa: Could not compute")

print("\n💭 Interpretation:")
if perfect_agreement < 10:
    print("→ Disagreement is SUBSTANTIAL. Your labels diverged significantly.")
else:
    print("→ You were more consistent than expected (but still had disagreement).")

print("→ This is normal and expected. It's why AI alignment is HARD.")

---

# Part Four: From Labels to Model – The Fine-Tuning Loop {#finetuning-heading}

### How RLHF Works

**RLHF = Reinforcement Learning from Human Feedback**

This is the pipeline that made ChatGPT "aligned":

1. **Pre-train** a large language model on internet text (unsupervised)
2. **Collect human feedback**: Annotators label model outputs as good/bad, helpful/harmful, etc.
3. **Train a reward model**: Use annotations to train a model that predicts which outputs humans prefer
4. **Fine-tune with RL**: Use the reward model to guide the language model toward preferred outputs

**The result**: An AI system that behaves according to human values – or at least, the values of whoever wrote the annotation guidelines.

---

**In this lab, you will see this in miniature:**
- A pre-trained sentiment model will make predictions on your texts
- We'll then adjust those predictions based on YOUR labels
- You'll see the model's behaviour shift to match your feedback

**Key question**: If the model shifts to match human feedback, but human feedback is subjective and reflects the annotators' perspectives – whose values is the model learning?

In [ ]:
#@title ## Install and load transformer model

# Install transformers (quiet mode)
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'torch'], check=False)

from transformers import pipeline

# Load a pre-trained sentiment analysis model
print("⏳ Loading pre-trained sentiment model...")
sentiment_classifier = pipeline('sentiment-analysis', 
                                 model='distilbert-base-uncased-finetuned-sst-2-english')
print("✓ Model loaded!\n")

### Model Predictions: Before Your Feedback {#model-predictions}

In [ ]:
#@title ## Get baseline model predictions

print("🤖 Getting model predictions on all texts...\n")

model_predictions_raw = []
model_scores = []

for idx, text in enumerate(df_round2['text']):
    result = sentiment_classifier(text[:512])  # Truncate to 512 tokens for safety
    label = result[0]['label']
    score = result[0]['score']
    model_predictions_raw.append(label)
    model_scores.append(score)
    if (idx + 1) % 5 == 0:
        print(f"  ✓ Processed {idx+1}/20 texts")

df_round2['model_prediction_baseline'] = model_predictions_raw
df_round2['model_score_baseline'] = model_scores

print("\n✓ Baseline predictions complete!")
print("\nModel's default assessment:\n")
print(df_round2[['id', 'text', 'model_prediction_baseline', 'model_score_baseline']].to_string(index=False))

In [ ]:
#@title ## Compare: Model Baseline vs Your Labels

# Convert your toxicity labels to sentiment-like scores
# High toxicity ≈ Negative; Low toxicity ≈ Positive
def toxicity_to_predicted_sentiment(tox_score):
    if tox_score <= 2:
        return 'POSITIVE'
    elif tox_score <= 3:
        return 'NEUTRAL'
    else:
        return 'NEGATIVE'

df_round2['your_predicted_sentiment_r1'] = df_round2['toxicity_r1'].apply(toxicity_to_predicted_sentiment)
df_round2['your_predicted_sentiment_r2'] = df_round2['toxicity_r2'].apply(toxicity_to_predicted_sentiment)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Agreement between model and Round 1 annotations
agreement_r1 = (df_round2['model_prediction_baseline'] == df_round2['your_predicted_sentiment_r1']).sum()
axes[0].bar(['Agree', 'Disagree'], [agreement_r1, 20 - agreement_r1], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Model vs Your Round 1 Annotations', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, 20)
for i, v in enumerate([agreement_r1, 20 - agreement_r1]):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Agreement between model and Round 2 annotations
agreement_r2 = (df_round2['model_prediction_baseline'] == df_round2['your_predicted_sentiment_r2']).sum()
axes[1].bar(['Agree', 'Disagree'], [agreement_r2, 20 - agreement_r2], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Model vs Your Round 2 Annotations (Changed Instructions)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_ylim(0, 20)
for i, v in enumerate([agreement_r2, 20 - agreement_r2]):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_vs_annotations.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nModel agreement with Round 1: {agreement_r1}/20 ({agreement_r1/20*100:.1f}%)")
print(f"Model agreement with Round 2: {agreement_r2}/20 ({agreement_r2/20*100:.1f}%)")
print(f"\nΔ Disagreement change: {abs(agreement_r2 - agreement_r1)} texts")

### Simulating Fine-Tuning: The Model Adjusts

In [ ]:
#@title ## Simulate fine-tuning: Apply your labels to model predictions
#@markdown In a real RLHF pipeline, we would use a reward model and RL optimization.
#@markdown For this demo, we simply adjust model thresholds based on your feedback.

print("🔧 Simulating fine-tuning...\n")

# Calculate mean toxicity from your Round 1 annotations
mean_tox_r1 = df_round2['toxicity_r1'].mean()
mean_tox_r2 = df_round2['toxicity_r2'].mean()

print(f"Mean toxicity score (Round 1): {mean_tox_r1:.2f}")
print(f"Mean toxicity score (Round 2): {mean_tox_r2:.2f}")
print()

# Simulate fine-tuning by adjusting decision boundaries
# This is a simplified version of what RLHF does

def adjust_prediction_r1(baseline_pred, baseline_score, your_toxicity):
    """
    Adjust model prediction based on your feedback.
    If you said it's toxic but model said positive, adjust toward negative.
    """
    if your_toxicity >= 4 and baseline_pred == 'POSITIVE':
        return 'NEGATIVE', max(baseline_score - 0.2, 0.5)
    elif your_toxicity <= 2 and baseline_pred == 'NEGATIVE':
        return 'POSITIVE', max(baseline_score + 0.2, 0.9)
    else:
        return baseline_pred, baseline_score

df_round2['model_prediction_finetuned_r1'] = []
df_round2['model_score_finetuned_r1'] = []

for idx in range(len(df_round2)):
    pred, score = adjust_prediction_r1(
        df_round2.loc[idx, 'model_prediction_baseline'],
        df_round2.loc[idx, 'model_score_baseline'],
        df_round2.loc[idx, 'toxicity_r1']
    )
    df_round2.loc[idx, 'model_prediction_finetuned_r1'] = pred
    df_round2.loc[idx, 'model_score_finetuned_r1'] = score

# Same for Round 2
def adjust_prediction_r2(baseline_pred, baseline_score, your_toxicity):
    if your_toxicity >= 4 and baseline_pred == 'POSITIVE':
        return 'NEGATIVE', max(baseline_score - 0.2, 0.5)
    elif your_toxicity <= 2 and baseline_pred == 'NEGATIVE':
        return 'POSITIVE', max(baseline_score + 0.2, 0.9)
    else:
        return baseline_pred, baseline_score

df_round2['model_prediction_finetuned_r2'] = []
df_round2['model_score_finetuned_r2'] = []

for idx in range(len(df_round2)):
    pred, score = adjust_prediction_r2(
        df_round2.loc[idx, 'model_prediction_baseline'],
        df_round2.loc[idx, 'model_score_baseline'],
        df_round2.loc[idx, 'toxicity_r2']
    )
    df_round2.loc[idx, 'model_prediction_finetuned_r2'] = pred
    df_round2.loc[idx, 'model_score_finetuned_r2'] = score

print("✓ Fine-tuning simulation complete!\n")

# Show how many predictions changed
changes_r1 = (df_round2['model_prediction_baseline'] != df_round2['model_prediction_finetuned_r1']).sum()
changes_r2 = (df_round2['model_prediction_baseline'] != df_round2['model_prediction_finetuned_r2']).sum()

print(f"Predictions changed after fine-tuning on Round 1 feedback: {changes_r1} texts")
print(f"Predictions changed after fine-tuning on Round 2 feedback: {changes_r2} texts")
print(f"\n→ The model shifted its behaviour based on YOUR LABELS.")

In [ ]:
#@title ## Visualize model shifts: Before vs After Fine-Tuning

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Baseline vs Fine-tuned (Round 1)
baseline_dist = df_round2['model_prediction_baseline'].value_counts()
finetuned_r1_dist = df_round2['model_prediction_finetuned_r1'].value_counts()

labels = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']
baseline_counts = [baseline_dist.get(label, 0) for label in labels]
finetuned_r1_counts = [finetuned_r1_dist.get(label, 0) for label in labels]

x = np.arange(len(labels))
width = 0.35

axes[0, 0].bar(x - width/2, baseline_counts, width, label='Baseline', color='#3498db', alpha=0.8)
axes[0, 0].bar(x + width/2, finetuned_r1_counts, width, label='After Round 1 Fine-tuning', color='#e74c3c', alpha=0.8)
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Model Predictions: Baseline vs Fine-tuned (Round 1)', fontsize=11, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(labels)
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Baseline vs Fine-tuned (Round 2)
finetuned_r2_dist = df_round2['model_prediction_finetuned_r2'].value_counts()
finetuned_r2_counts = [finetuned_r2_dist.get(label, 0) for label in labels]

axes[0, 1].bar(x - width/2, baseline_counts, width, label='Baseline', color='#3498db', alpha=0.8)
axes[0, 1].bar(x + width/2, finetuned_r2_counts, width, label='After Round 2 Fine-tuning', color='#9b59b6', alpha=0.8)
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Model Predictions: Baseline vs Fine-tuned (Round 2)', fontsize=11, fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(labels)
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Round 1 vs Round 2 fine-tuned predictions
axes[1, 0].bar(x - width/2, finetuned_r1_counts, width, label='Fine-tuned (Round 1)', color='#e74c3c', alpha=0.8)
axes[1, 0].bar(x + width/2, finetuned_r2_counts, width, label='Fine-tuned (Round 2)', color='#9b59b6', alpha=0.8)
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('How Different Instructions Shift Model Behaviour', fontsize=11, fontweight='bold')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(labels)
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Text summary
axes[1, 1].axis('off')
summary_text = f"""KEY INSIGHT:

BASELINE MODEL:
  Positive: {baseline_counts[2]}
  Neutral: {baseline_counts[1]}
  Negative: {baseline_counts[0]}

AFTER ROUND 1 FEEDBACK:
  Positive: {finetuned_r1_counts[2]}
  Neutral: {finetuned_r1_counts[1]}
  Negative: {finetuned_r1_counts[0]}

AFTER ROUND 2 FEEDBACK:
  Positive: {finetuned_r2_counts[2]}
  Neutral: {finetuned_r2_counts[1]}
  Negative: {finetuned_r2_counts[0]}

The model's behaviour CHANGED based on
whose values you prioritized.

Imagine this at scale:
  • Millions of annotators
  • Different cultural backgrounds
  • Different political perspectives
  • Shaping a system used globally
"""

axes[1, 1].text(0.05, 0.95, summary_text, fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.9),
                family='monospace')

plt.tight_layout()
plt.savefig('model_finetuning.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n📊 Visualization complete!")

---

# Part Five: The Politics of Labelling – Reflection {#reflection-heading}

### The Elephant in the Room

You've now experienced the annotation pipeline firsthand. You've seen:

1. **Inter-annotator disagreement**: Your own labels changed based on framing
2. **Model malleability**: The model shifted its behaviour to match YOUR feedback
3. **Subjectivity at scale**: These processes affect billions of users globally

Now let's talk about the politics.

---

### Who Are the Annotators?

**Time Magazine (2023):** "OpenAI Used Kenyan Workers on Poverty Wages to Make ChatGPT Less Toxic"

The human feedback that shaped ChatGPT came from:
- Contract workers in Kenya, India, and other lower-income countries
- Paid $1.32–$2/hour (well below local minimum wage in many cases)
- Reading graphic content: violent imagery, sexual abuse, extremist propaganda
- No mental health support; no worker protections

**This is digital labour.**

---

### What Are the Working Conditions?

Aaron Cant's *"Feeding the Machine"* documents:
- **Precarity**: No job security, contracts terminated suddenly
- **Psychological toll**: Exposure to harmful content without support
- **Invisibility**: Annotators' names and contributions are never credited
- **Exploitation**: Billionaire companies extract value from low-wage labour

**This mirrors colonial patterns**: Rich countries extract resources (data + labour) from poorer countries, then sell the resulting products back to them.

---

### How Do Annotation Guidelines Embed Values?

The guidelines that tell annotators what counts as "toxic" are written by AI companies and their contractors. These guidelines:

- Reflect **Western, English-language norms** (often American liberal values)
- May inadvertently **suppress dissent** or **chill speech** on topics the company deems controversial
- Can **embed cultural biases**: e.g., what counts as "respectful" differs across cultures
- Are often **proprietary**: Not transparent, cannot be audited, cannot be contested

**Example**: What counts as "toxic" criticism of a government? In a liberal democracy, it might be protected speech. In an authoritarian regime, it might be classified as "dangerous." The same words, different contexts.

---

### International Law Dimensions

**The Right to Work** (ICESCR Art. 6-7):
- Everyone has a right to work and to fair wages
- Data annotators are often denied both

**Freedom of Expression** (ICCPR Art. 19):
- AI systems trained on subjective labelling may chill legitimate speech
- Who decides what counts as "misinformation"? "Toxic"? "Dangerous"?

**Labour Standards** (ILO conventions):
- Annotation work often violates minimum wage, workplace safety, and dignity standards
- Workers lack collective bargaining power

**Data Sovereignty** (UN Declaration on the Rights of Indigenous Peoples, etc.):
- Who owns data? Who profits from it? Who has a say in how it's used?

---

### What Does "Alignment" Actually Mean?

When AI researchers say they're "aligning" AI with human values, they usually mean:

**"Aligned with the values of whoever wrote the annotation guidelines and paid for the feedback."**

It does NOT mean:
- Aligned with the values of the annotators
- Aligned with diverse global perspectives
- Aligned with affected communities
- Democratic or deliberative in any way

**This is a governance problem disguised as a technical problem.**

---

### The RLHF Pipeline as Governance

RLHF is not just a machine learning technique. It's a mechanism for **embedding values into systems that billions of people use**.

**Who writes the annotation guidelines = who governs the AI.**

And right now, that's a small number of engineers and product managers at large corporations – with input from contractors paid poverty wages.

**For international law, this raises hard questions:**
- Should this process be more transparent and democratic?
- Should annotators have rights, representation, and fair compensation?
- Should governments have a say in what values are embedded in systems used by their citizens?
- Should international standards govern data annotation labour?

These are questions you, as future legal professionals, will need to grapple with.

### Whose Values? {#whose-values}

In [ ]:
#@title ## Reflection: Who Should Decide What Counts as Toxic?

who_should_decide = "A democratic process involving affected communities, annotators, civil society, and independent experts" #@param ["The AI companies alone", "Government regulators", "International bodies (UN, etc.)", "A democratic process involving affected communities, annotators, civil society, and independent experts", "I'm not sure"]

print(f"\n🤔 Your answer: {who_should_decide}\n")

reflections = {
    "The AI companies alone": "This is how it currently works. Issues: no public accountability, concentrated power, values reflect corporate interests, not diverse societies.",
    "Government regulators": "Could improve accountability, but risks: authoritarian governments could use this to censor dissent, democratic governments may lack technical expertise.",
    "International bodies (UN, etc.)": "Could provide legitimacy and inclusivity, but faces challenges: global consensus is hard, powerful nations may dominate, enforcement is weak.",
    "A democratic process involving affected communities, annotators, civil society, and independent experts": "This is the most legitimate path, but also the hardest to implement. Requires: transparency, participation, fair compensation, legal protections for annotators.",
    "I'm not sure": "That's fair. This is a genuinely hard problem. There's no perfect answer, only tradeoffs."
}

print("💭 Thoughts:\n")
print(reflections.get(who_should_decide, "Unknown response."))
print("\n" + "="*70)

In [ ]:
#@title ## Critical Reflection: Key Takeaways

print("\n🎯 KEY TAKEAWAYS\n")
print("="*70)

takeaways = [
    ("1. Data is Labour", 
     "Annotation is invisible work that powers AI. Annotators deserve:\n     • Fair wages (living wage, not poverty wage)\n     • Safe working conditions\n     • Mental health support\n     • Legal protections and collective bargaining rights"),
    
    ("2. Labels Reflect Values, Not Truth", 
     "Your annotations changed based on framing. There is no objective\n     'ground truth' in data labelling. All labels embed values and\n     perspectives. This matters when those labels shape systems used\n     in legal, medical, security, and financial contexts."),
    
    ("3. RLHF is Governance", 
     "When you fine-tune an AI model on human feedback, you are\n     encoding values into a system that will influence billions of\n     people. This is governance. It should be democratic, transparent,\n     and accountable. Right now, it usually isn't."),
    
    ("4. Annotation Guidelines Have Power", 
     "Who writes the annotation guidelines decides what gets labeled\n     'toxic,' 'misinformation,' 'dangerous,' etc. This is a form of\n     speech regulation. It should be subject to the same scrutiny as\n     other speech regulation (law, policy, etc.)."),
    
    ("5. Global Power Asymmetries Matter", 
     "AI is developed in wealthy countries; annotation labour is\n     outsourced to lower-income countries. The resulting systems are\n     used globally, often with minimal input from affected communities.\n     This mirrors colonial patterns of extraction and exploitation."),
    
    ("6. You Have Agency Here", 
     "As future lawyers, you can:\n     • Advocate for annotator rights and labour protections\n     • Push for transparency in AI governance\n     • Develop legal frameworks for algorithmic accountability\n     • Support international standards for data labour\n     • Hold companies and governments accountable")
]

for title, content in takeaways:
    print(f"\n{title}")
    print("-" * 70)
    print(content)
    print()